# PICK: Semi-Supervised Segmentation for BHSD (Fold 0) on Kaggle GPU

### Hướng dẫn thiết lập môi trường Kaggle:
1. **Notebook Settings (Cột tùy chọn bên phải màn hình):**
   - **Accelerator**: Chọn **GPU T4 x 2** hoặc **GPU P100** (16GB VRAM).
   - **Internet**: Bật sang **ON** (để clone mã nguồn GitHub và cài đặt thư viện).
2. **Thêm Dataset vào Kaggle Notebook:**
   - Bấm nút **+ Add Input** (hoặc **+ Add Data**) ở góc trên bên phải màn hình.
   - Chọn **Your Datasets** -> Tìm chọn dataset BHSD Fold 0 đã upload (ví dụ: `BHSD Fold 0` / dataset `bhsd-fold0`).
   - Dataset sẽ tự động được mount tại thư mục `/kaggle/input/`.
3. **Mã nguồn:**
   - Clone trực tiếp từ GitHub (`https://github.com/NguyenVanCongThanh/PICK_BHSD.git`).

### Bước 1: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q medpy tensorboardX h5py SimpleITK nibabel

### Bước 2: Clone mã nguồn PICK từ GitHub
Mã nguồn sẽ được clone về `/kaggle/working/PICK`.

> **Lưu ý:** Nếu repository của bạn ở chế độ **Private**, hãy tạo **Personal Access Token (PAT)** trên GitHub (Settings -> Developer Settings -> Personal access tokens) và điền vào biến `GITHUB_TOKEN` bên dưới. Nếu repo là Public, bạn chỉ cần giữ nguyên.

In [ ]:
import os

# Cấu hình GitHub Repository
REPO_NAME = "NguyenVanCongThanh/PICK_BHSD"
GITHUB_TOKEN = ""  # Điền token ghp_xxx nếu repo là Private, nếu Public thì để trống

if GITHUB_TOKEN.strip():
    GITHUB_REPO_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_NAME}.git"
else:
    GITHUB_REPO_URL = f"https://github.com/{REPO_NAME}.git"

WORK_DIR = "/kaggle/working/PICK"

if not os.path.exists(WORK_DIR):
    print(f"[*] Đang clone repository {REPO_NAME}...")
    !git clone {GITHUB_REPO_URL} {WORK_DIR}
else:
    print(f"[OK] Repository đã tồn tại tại {WORK_DIR}. Đang cập nhật git pull...")
    !git -C {WORK_DIR} pull

%cd {WORK_DIR}
!git log -1 --oneline
!ls -la

### Bước 3: Định vị Dataset BHSD Fold 0 từ Kaggle Input & Thiết lập Dữ liệu
Sử dụng trực tiếp dataset `BHSD Fold 0` đã được upload lên Kaggle Dataset và gắn vào Notebook (`/kaggle/input/`).

> **Ưu điểm & Cơ chế:**
> - **Không cần tải qua mạng**: Tiết kiệm 20-30 phút tải ~20GB file so với Hugging Face hay Google Drive.
> - **Tự động nhận diện cấu trúc**: Kiểm tra xem dataset đã giải nén sẵn trong `/kaggle/input` hay cần giải nén từ file `BHSD_fold0.tar`.
> - **Giải nén cực nhanh**: Chỉ mất ~1-2 phút giải nén từ bộ nhớ đĩa cục bộ sang `/kaggle/working`.
> - **Bảo đảm toàn vẹn**: Tự động định vị chính xác thư mục chứa `data/` (các file `.h5`) và `splits/` (`fold0_train.txt`, `fold0_val.txt`).

In [ ]:
import os, glob, shutil

# ==============================================================================
# ĐỊNH VỊ DATASET BHSD FOLD 0 TỪ KAGGLE INPUT
# ==============================================================================
INPUT_DIR = "/kaggle/input"
WORKING_DIR = "/kaggle/working"

def find_bhsd_root(base_dir):
    """Tìm thư mục chứa cả thư mục 'data' và 'splits' hoặc file split fold0."""
    if not os.path.exists(base_dir):
        return None
    for root, dirs, files in os.walk(base_dir):
        if "data" in dirs and ("splits" in dirs or "fold0_train.txt" in files):
            return root
    return None

DATA_ROOT = None

# 1. Kiểm tra nếu dataset đã có sẵn (hoặc Kaggle đã tự giải nén) trong /kaggle/input
print("[*] Đang quét dataset BHSD Fold 0 trong /kaggle/input...")
DATA_ROOT = find_bhsd_root(INPUT_DIR)

if DATA_ROOT:
    print(f"[OK] Tìm thấy dataset sẵn sàng trong Kaggle Input tại: {DATA_ROOT}")
    print("[*] Sử dụng trực tiếp từ Kaggle Input (tiết kiệm dung lượng đĩa cho /kaggle/working).")
else:
    # 2. Kiểm tra nếu dataset đã giải nén trong /kaggle/working từ lần chạy trước
    DATA_ROOT = find_bhsd_root(WORKING_DIR)
    if DATA_ROOT:
        print(f"[OK] Dataset đã tồn tại sẵn trong /kaggle/working tại: {DATA_ROOT}")
    else:
        # 3. Tìm file BHSD_fold0.tar trong /kaggle/input
        print("[*] Đang tìm file nén BHSD_fold0.tar trong /kaggle/input...")
        all_tars = glob.glob(os.path.join(INPUT_DIR, "**", "*.tar"), recursive=True)
        matching_tars = [p for p in all_tars if "bhsd" in os.path.basename(p).lower() or "fold0" in os.path.basename(p).lower()]
        tar_path = matching_tars[0] if matching_tars else (all_tars[0] if all_tars else None)

        if not tar_path or not os.path.exists(tar_path):
            available_inputs = os.listdir(INPUT_DIR) if os.path.exists(INPUT_DIR) else []
            raise FileNotFoundError(
                f"[X] Không tìm thấy file BHSD_fold0.tar trong {INPUT_DIR}!\n"
                f"    Danh sách thư mục hiện có trong /kaggle/input: {available_inputs}\n"
                "    Vui lòng kiểm tra:\n"
                "    1. Bấm nút '+ Add Input' (hoặc '+ Add Data') ở góc phải trên Kaggle Notebook.\n"
                "    2. Chọn dataset 'BHSD Fold 0' bạn đã upload lên Kaggle."
            )

        tar_size_gb = os.path.getsize(tar_path) / (1024**3)
        print(f"[OK] Tìm thấy file dataset: {tar_path} ({tar_size_gb:.2f} GB)")
        print(f"[*] Đang giải nén {os.path.basename(tar_path)} vào {WORKING_DIR} (khoảng 1-2 phút)...\n")

        # Giải nén trực tiếp vào /kaggle/working
        !tar -xf "{tar_path}" -C {WORKING_DIR}
        print("\n[OK] Giải nén hoàn tất!")

        # 4. Xác định lại DATA_ROOT sau khi giải nén
        DATA_ROOT = find_bhsd_root(WORKING_DIR)
        if not DATA_ROOT:
            raise RuntimeError(
                f"[X] Giải nén xong nhưng không tìm thấy cấu trúc 'data' và 'splits' trong {WORKING_DIR}!\n"
                f"    Nội dung {WORKING_DIR}: {os.listdir(WORKING_DIR)}"
            )

print(f"\n[SUCCESS] Đường dẫn DATA_ROOT đã sẵn sàng: {DATA_ROOT}")

### Bước 4: Kiểm tra GPU & Tính toàn vẹn của Dataset (Integrity Check)
Ô này kiểm tra thông số phần cứng GPU và xác thực toàn bộ cấu trúc dữ liệu của Fold 0:
- Danh sách train: `splits/fold0_train.txt` (2,132 ca)
- Danh sách validation: `splits/fold0_val.txt` (38 ca)
- Danh sách labeled: `train_labeled.txt` (154 ca)
- Kho file volume HDF5: `data/*.h5` (đầy đủ 2,170 file)

In [ ]:
import torch, os, glob

print("=" * 60)
print("=== 1. KIỂM TRA PHẦN CỨNG GPU ===")
print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Thiết bị:", torch.cuda.get_device_name(0))
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    print(f"Bộ nhớ VRAM: {vram_gb} GB")
else:
    print("[CẢNH BÁO] Không tìm thấy GPU! Hãy chọn GPU T4 hoặc P100 ở Notebook Settings.")

print("\n" + "=" * 60)
print("=== 2. KIỂM TRA TÍNH TOÀN VẸN CỦA DATASET FOLD 0 ===")
print("Đường dẫn DATA_ROOT:", DATA_ROOT)
assert DATA_ROOT and os.path.exists(DATA_ROOT), f"Không tồn tại thư mục {DATA_ROOT}"

train_split = os.path.join(DATA_ROOT, "splits", "fold0_train.txt")
val_split = os.path.join(DATA_ROOT, "splits", "fold0_val.txt")
labeled_split = os.path.join(DATA_ROOT, "train_labeled.txt")
if not os.path.exists(labeled_split):
    alt_labeled = os.path.join(DATA_ROOT, "splits", "train_labeled.txt")
    if os.path.exists(alt_labeled):
        labeled_split = alt_labeled
data_dir = os.path.join(DATA_ROOT, "data")

for p, name in [(train_split, "fold0_train.txt"), (val_split, "fold0_val.txt"), (data_dir, "data directory")]:
    assert os.path.exists(p), f"[LỖI] Thiếu {name} tại {p}"
    
with open(train_split) as f:
    train_cases = [line.strip() for line in f if line.strip()]
with open(val_split) as f:
    val_cases = [line.strip() for line in f if line.strip()]

labeled_count = 0
if os.path.exists(labeled_split):
    with open(labeled_split) as f:
        labeled_count = len([line.strip() for line in f if line.strip()])

h5_count = len(glob.glob(os.path.join(data_dir, "*.h5")))

print(f"[+] Số mẫu huấn luyện (train):      {len(train_cases)} (Kỳ vọng: 2132)")
print(f"[+] Số mẫu kiểm tra (val):          {len(val_cases)} (Kỳ vọng: 38)")
print(f"[+] Số mẫu có nhãn chuẩn (labeled): {labeled_count} (Kỳ vọng: 154)")
print(f"[+] Tổng số file .h5 trong data/:    {h5_count} (Kỳ vọng: 2170)")

# Xác nhận tính toàn vẹn mẫu ngẫu nhiên
sample_h5 = os.path.join(data_dir, f"{train_cases[0]}.h5")
assert os.path.exists(sample_h5), f"Không tìm thấy file mẫu {sample_h5}"

print("\n[OK] TẤT CẢ DỮ LIỆU ĐÃ ĐẦY ĐỦ VÀ SẴN SÀNG ĐỂ CHẠY HUẤN LUYỆN!")
print("=" * 60)

### Bước 5: (Khuyên dùng) Khởi chạy TensorBoard theo dõi trực quan Real-time
Chạy ô này **TRƯỚC** khi bấm chạy cell huấn luyện (Bước 6 hoặc Bước 7). Giao diện tương tác TensorBoard sẽ được nhúng trực tiếp ngay trong Notebook:
- **Theo dõi biểu đồ Loss**: Biểu đồ hàm mất mát tổng thể, Main loss, MIM reconstruction loss, Aux loss và Learning rate.
- **Theo dõi độ đo Dice**: mDice kiểm định và Dice chi tiết từng phân loại xuất huyết (EDH, IPH, IVH, SAH, SDH) được cập nhật sau mỗi 200 bước.
- **Mẹo hữu ích**: Bấm vào biểu tượng bánh răng ⚙️ ở góc trên bên phải giao diện TensorBoard và tích chọn **Reload data** (mỗi 30s) để tự động cập nhật biểu đồ mà không cần bấm Refresh thủ công.

In [ ]:
# ==============================================================================
# KHỞI CHẠY TENSORBOARD THEO DÕI TIẾN TRÌNH TRỰC QUAN
# ==============================================================================
import os
os.makedirs("/kaggle/working/experiments", exist_ok=True)

%reload_ext tensorboard
%tensorboard --logdir /kaggle/working/experiments --port 6006


### Bước 6: Huấn luyện PICK - Binary Segmentation (Fold 0)

**Chi tiết tham số:**
- `--fold 0`: Chỉ định Fold 0 trong 5-fold cross validation.
- `--labelnum 154`: Sử dụng đầy đủ 154 ca có nhãn của Fold 0 (~7.2%). Nếu bạn muốn thử nghiệm bài toán ít nhãn hơn (ví dụ benchmark ~10% của tập labeled), bạn có thể đặt `--labelnum 15`.
- `--max_samples 2132`: Sử dụng toàn bộ 2,132 ca của tập train Fold 0.
- `--batch_size 4` & `--labeled_bs 2`: Mỗi batch gồm 2 ảnh labeled + 2 ảnh unlabeled (vừa vặn 16GB VRAM của T4/P100).
- `--pre_max_iteration 4000`: 4,000 bước pre-training.
- `--self_max_iteration 12000`: 12,000 bước self-training.
- Thời gian chạy: ~50-70 phút trên Kaggle GPU.

In [ ]:
# === HUẤN LUYỆN BINARY SEGMENTATION (FOLD 0) ===
FOLD_ID = 0
EXP_NAME = "BHSD_binary_PICK_fold0"
EXP_DIR = "/kaggle/working/experiments"

LABEL_NUM = 154       # Đầy đủ 154 ca có nhãn của Fold 0 (hoặc chọn 15 nếu muốn benchmark ít nhãn)
MAX_SAMPLES = 2132    # Toàn bộ 2132 ca train của Fold 0
BATCH_SIZE = 4
LABELED_BS = 2
PRE_ITERS = 4000
SELF_ITERS = 12000

!python -u BHSD_binary_PICK_train.py \
    --root_path {DATA_ROOT} \
    --exp {EXP_NAME} \
    --exp_dir {EXP_DIR} \
    --fold {FOLD_ID} \
    --model VNet \
    --labelnum {LABEL_NUM} \
    --max_samples {MAX_SAMPLES} \
    --batch_size {BATCH_SIZE} \
    --labeled_bs {LABELED_BS} \
    --pre_max_iteration {PRE_ITERS} \
    --self_max_iteration {SELF_ITERS} \
    --gpu 0 \
    --resume


### Bước 7: (Tùy chọn) Huấn luyện PICK - Multi-class Segmentation (6 classes)
Phân đoạn 5 phân loại xuất huyết não chi tiết (EDH, IPH, IVH, SAH, SDH) + Background.
Bỏ comment ô dưới nếu muốn chạy huấn luyện Multi-class cho Fold 0.

In [ ]:
# === HUẤN LUYỆN MULTI-CLASS (FOLD 0) ===
# FOLD_ID = 0
# EXP_NAME = "BHSD_multiclass_PICK_fold0"
# EXP_DIR = "/kaggle/working/experiments"

# !python -u BHSD_multiclass_PICK_train.py \
#     --root_path {DATA_ROOT} \
#     --exp {EXP_NAME} \
#     --exp_dir {EXP_DIR} \
#     --fold {FOLD_ID} \
#     --model VNet \
#     --labelnum 154 \
#     --max_samples 2132 \
#     --batch_size 4 \
#     --labeled_bs 2 \
#     --pre_max_iteration 4000 \
#     --self_max_iteration 12000 \
#     --gpu 0 \
#     --resume


### Bước 8: Nén và Xuất Kết quả Model Checkpoints / Logs
Sau khi quá trình huấn luyện hoàn tất:
1. Ô này sẽ nén thư mục `/kaggle/working/experiments` (bao gồm model trọng số tốt nhất `VNet_best_model.pth`, log TensorBoard, và log huấn luyện `train.log`) thành file `bhsd_pick_fold0_results.tar.gz`.
2. Bạn có thể tải file này trực tiếp về máy tính tại mục **Output** (ở góc dưới bên phải giao diện Kaggle Notebook).

In [ ]:
import os

OUTPUT_ARCHIVE = "/kaggle/working/bhsd_pick_fold0_results.tar.gz"
EXP_DIR = "/kaggle/working/experiments"

if os.path.exists(EXP_DIR):
    print("[*] Đang nén toàn bộ kết quả mô hình và logs...")
    !tar -czvf {OUTPUT_ARCHIVE} -C {EXP_DIR} .
    size_mb = round(os.path.getsize(OUTPUT_ARCHIVE) / (1024**2), 2)
    print(f"\n[OK] Đã nén thành công file kết quả: {OUTPUT_ARCHIVE} ({size_mb} MB)")
    print("[*] File đã sẵn sàng tải về tại tab Output bên phải màn hình!")
else:
    print(f"[!] Chưa tìm thấy thư mục {EXP_DIR}. Vui lòng kiểm tra lại quá trình huấn luyện.")